# Test code 

In [ ]:
%pip install --upgrade --quiet azure-cosmos langchain-openai langchain-community

In [1]:
OPENAI_API_KEY = "CrMbXFQGT3fpFaEolF2U69LczMQfe0zNmYF3N23pfBCNOdc7BVBaJQQJ99AJACfhMk5XJ3w3AAABACOGNgR3"
OPENAI_API_TYPE = "azure"
OPENAI_API_VERSION = "2023-05-15"
OPENAI_API_BASE = "https://kthapikey.openai.azure.com"
OPENAI_EMBEDDINGS_MODEL_NAME = "text-embedding-3-large"
OPENAI_EMBEDDINGS_MODEL_DEPLOYMENT = "text-embedding-3-large"

In [2]:
from langchain_community.document_loaders import TextLoader

# Load the PDF
loader = TextLoader("./data/Q.txt")
data = loader.load()

In [3]:
data

[Document(metadata={'source': './data/Q.txt'}, page_content='    1. Fråga: Varför är min elanvändning hög om uppvärmningen inte är eldriven?\nSvar: Andra apparater och elektroniska enheter, som belysning, köksutrustning och underhållningssystem, kan vara ansvariga för hög elanvändning.\n\n    2. Fråga: Vilka apparater kan vara de största elslukarna i en lägenhet?\nSvar: Vanligtvis är uppvärmning, kylning, varmvattenberedare, tvättmaskin och torktumlare de största elkonsumenterna.\n\n    3. Fråga: Hur kan jag identifiera specifika apparater som bidrar till hög elanvändning?\nSvar: Använd energimätare eller smarta eluttag för att övervaka förbrukningen av individuella apparater och identifiera energislukare.\n\n    4. Fråga: Kan felaktig isolering påverka elanvändningen?\nSvar: Ja, bristfällig isolering kan tvinga uppvärmnings- och kylsystem att arbeta hårdare, vilket kan öka elförbrukningen.\n\n    5. Fråga: Hur påverkar elanvändningen mitt elpris?\nSvar: Högre elanvändning resulterar v

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

In [ ]:
indexing_policy = {
    "indexingMode": "consistent",
    "includedPaths": [{"path": "/*"}],
    "excludedPaths": [{"path": '/"_etag"/?'}],
    "vectorIndexes": [{"path": "/embedding", "type": "quantizedFlat"}],
}

vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/embedding",
            "dataType": "float32",
            "distanceFunction": "cosine",
            "dimensions": 3072,
        }
    ]
}

In [6]:
from azure.cosmos import CosmosClient, PartitionKey
from langchain_community.vectorstores.azure_cosmos_db_no_sql import (
    AzureCosmosDBNoSqlVectorSearch,
)
from langchain_openai import AzureOpenAIEmbeddings

In [7]:
HOST = "https://sparaknowledgebase.documents.azure.com:443/"
KEY = "FxdVcxDFLvWfufB2LFlEEw5kV6ADsMPH8FaNsLgAFnkqkoG5fD1EhZcYrqSLapluRb1RmO7pJOXSACDbSio1Qw=="

In [8]:
cosmos_client = CosmosClient(HOST, KEY)
database_name = "sparadocuments"
container_name = "sparacontainer"
partition_key = PartitionKey(path="/id")
cosmos_container_properties = {"partition_key": partition_key}
cosmos_database_properties = {"id": database_name}

openai_embeddings = AzureOpenAIEmbeddings(
    azure_deployment=OPENAI_EMBEDDINGS_MODEL_DEPLOYMENT,
    api_version=OPENAI_API_VERSION,
    azure_endpoint=OPENAI_API_BASE,
    openai_api_key=OPENAI_API_KEY,
)

In [9]:
database = cosmos_client.get_database_client(database_name)
container = database.get_container_client(container_name)

In [10]:
vector_search = AzureCosmosDBNoSqlVectorSearch(
    cosmos_client = cosmos_client , 
    database_name=database_name , 
    container_name=container_name , 
    embedding=openai_embeddings , 
    vector_embedding_policy=vector_embedding_policy,
    indexing_policy=indexing_policy,
    cosmos_container_properties=cosmos_container_properties, 
    cosmos_database_properties=cosmos_database_properties
)

CosmosHttpResponseError: (BadRequest) Message: {"Errors":["The Vector Indexing Policy's path::/AzureOpenAIEmbeddings not matching in Embedding's path."]}
ActivityId: 5a377475-31ab-42f8-ac65-978720c81aa1, Request URI: /apps/d26640e5-33e4-43af-b997-dfd51c386334/services/43d16bfc-dbc9-4359-8f8f-22d5008b34f0/partitions/ad2bfa6b-6329-445f-84d8-99ac91356e5f/replicas/133758522068149312p, RequestStats: , SDK: Microsoft.Azure.Documents.Common/2.14.0
Code: BadRequest
Message: Message: {"Errors":["The Vector Indexing Policy's path::/AzureOpenAIEmbeddings not matching in Embedding's path."]}
ActivityId: 5a377475-31ab-42f8-ac65-978720c81aa1, Request URI: /apps/d26640e5-33e4-43af-b997-dfd51c386334/services/43d16bfc-dbc9-4359-8f8f-22d5008b34f0/partitions/ad2bfa6b-6329-445f-84d8-99ac91356e5f/replicas/133758522068149312p, RequestStats: , SDK: Microsoft.Azure.Documents.Common/2.14.0

In [ ]:
vector_search

In [ ]:
#document_id_list = vector_search.add_documents(documents=docs)

In [ ]:
document_id_list

In [ ]:
type(openai_embeddings.embed_query(query))

In [ ]:
        items = list(
            self._container.query_items(
                query=query, parameters=parameters, enable_cross_partition_query=True
            )
        )

In [11]:
query = "Fråga: Kan jag få professionell hjälp för att genomföra en energieffektivitetskontroll i min lägenhet?"
embedding_vector = openai_embeddings.embed_query(query)
results = vector_search.similarity_search_with_score(query )

In [ ]:
results

In [ ]:
vector_search.max_marginal_relevance_search_by_vector(embedding_vector , pre_filter=None , with_embedding=None)

In [ ]:
results

In [12]:
vector_search.similarity_search(query , k = 5 , kind = 'vector-hnsw')

[Document(metadata={'source': './data/Q.txt'}, page_content='5. Fråga: Hur påverkar elanvändningen mitt elpris?\nSvar: Högre elanvändning resulterar vanligtvis i högre elkostnader på din elräkning.\n\n    6. Fråga: Finns det en koppling mellan vädret och min elanvändning?\nSvar: Ja, extrema temperaturer kan påverka kyl- eller värmesystemets arbetsbelastning och därmed öka elanvändningen.\n\n    7. Fråga: Kan jag minska elanvändningen genom att använda energieffektiva apparater?\nSvar: Ja, investera i apparater med hög energieffektivitet, som Energy Star-märkta produkter, för att minska elkostnaderna.\n\n    8. Fråga: Vilka åtgärder kan jag vidta för att minska elanvändningen i min lägenhet?\nSvar: Använd energisparande lampor, stäng av elektronik när de inte används, och överväg att använda termostater och timerkontroller.\n\n    9. Fråga: Kan jag få professionell hjälp för att genomföra en energieffektivitetskontroll i min lägenhet?\nSvar: Ja, energiexperter kan genomföra en energiutv

In [ ]:
vector_search.similarity_search(query , k = 5 , kind = 'vector-ivf')

# Function test

In [ ]:
import importlib

In [ ]:
import json 

main_config = json.load(open('main_config.json'))

In [ ]:
from knowledge_base.main_KB import *

In [ ]:
azure_blob_obj = Azure_Blob(account_url = main_config['account_url'] , container_name_blob = main_config['container_name_blob'])

In [ ]:
azure_container_client = azure_blob_obj.setup_blob_connection()

In [ ]:
azure_files_list = azure_blob_obj.get_document_list(azure_container_client)

In [ ]:
vector_database_obj = VectorDataBase(indexing_policy = main_config['indexing_policy'] , 
                                     vector_embedding_policy = main_config['vector_embedding_policy'] , 
                                     database_name = main_config['database_name'] , 
                                     container_name = main_config['container_name'])

In [ ]:
vector_database_obj.setup_connection()

In [ ]:
documents_present = vector_database_obj.get_document_source()

In [ ]:
directory_obj = Directory(vector_search = vector_database_obj.vector_search)

In [ ]:
knowledge_base_update_files = directory_obj.identify_documents_not_present(knowledge_base_data = documents_present , azure_file_list_names = azure_files_list)

In [ ]:
for i in knowledge_base_update_files : 
    azure_blob_obj.download_file_local(azure_container_client , main_config['temp_folder_download'] , i)
    if i == 'links_for_scrape.xlsx' : 
        id_list , excel_file = directory_obj.reading_URLS_for_scrape(main_config['temp_folder_download'] , 'links_for_scrape.xlsx')
        
        directory_path = Path.joinpath(Path().resolve() , main_config['temp_folder_download'])
        file_path = Path.joinpath(directory_path , 'links_for_scrape.xlsx')
        excel_file.to_excel(file_path , index=False)
        azure_blob_obj.upload_file_local(azure_container_client , 'links_for_scrape.xlsx' , main_config['temp_folder_download'])
        os.remove(file_path)
    else: 
        azure_blob_obj.download_file_local(azure_container_client , main_config['temp_folder_download']  , i)
        id_list = directory_obj.reading_file(main_config['temp_folder_download']  , i)
        
    print('Knowledge Base updated for ' + i)

# Retrieval 

In [13]:
from knowledge_base.main_KB import *

In [14]:
import json 

main_config = json.load(open('main_config.json'))

In [15]:
vector_database_obj = VectorDataBase(indexing_policy = main_config['indexing_policy'] , 
                                     vector_embedding_policy = main_config['vector_embedding_policy'] , 
                                     database_name = main_config['database_name'] , 
                                     container_name = main_config['container_name'])

In [16]:
vector_database_obj.setup_connection()

Connection to Vector Database established.


In [17]:
vector_search = vector_database_obj.vector_search
embeddings = vector_database_obj.openai_embeddings

In [18]:
from context_retrieval.main_context_retrieval import *

In [19]:
context_retreival_obj = RetrievalText(vector_search = vector_database_obj.vector_search , 
                                        openai_embeddings = vector_database_obj.openai_embeddings)

print('Context Retreival has been established')

Context Retreival has been established


In [20]:
query = "Fråga: Kan jag få professionell hjälp för att genomföra en energieffektivitetskontroll i min lägenhet?"

In [21]:
docs = context_retreival_obj.search_text_with_score(query)

'SimilarityScore'


In [ ]:
texts